# Chapter 5: Five LP Applications with CVXPY
Run each example independently. If needed, install CVXPY with `%pip install cvxpy`.
CVXPY selects the solver automatically. The slides use the saved outputs in this notebook.
Multiple optimal solutions may produce different allocations on other solver versions.
Displayed values are rounded. Rounding does not enforce integer constraints.


## 1. Portfolio Selection

An investor wants to allocate the entire investment budget across four assets to maximize expected annual return. Use the following return estimates and risk indices.

| Asset | Variable | Expected annual return | Risk index |
|---|---|---:|---:|
| Equity fund (stocks) | S | 10% | 80 |
| Bonds | B | 3% | 20 |
| Income fund | I | 4% | 30 |
| Money market fund | M | 1% | 0 |

**Investment policies**

- Invest the entire budget. Each variable represents a fraction of the total budget, so the fractions must sum to 1.
- The portfolio's weighted average risk index must not exceed 40.
- Invest no more than 75% in the equity fund.
- The fraction invested in the income fund must be at least as large as the fraction invested in bonds.
- Invest between 10% and 30% in the money market fund.
- All investment fractions must be nonnegative.

**Your task**

1. Formulate an LP that maximizes expected annual return.
2. Solve the model using CVXPY.
3. Report each investment share as a percentage and the portfolio's expected annual return. Check the investment policies using the unrounded solution.

The objective is an annual return expressed as a fraction. The code converts it and the investment shares to percentages for display.


In [ ]:
import cvxpy as cp

# 1. Decision variables
S = cp.Variable()
B = cp.Variable()
I = cp.Variable()
M = cp.Variable()

# 2. Objective function
objective = cp.Maximize(0.1*S + 0.03*B + 0.04*I + 0.01*M)

# 3. Constraints
constraints = [
    S + B + I + M == 1,
    80*S + 20*B + 30*I <= 40,
    S <= 0.75,
    I - B >= 0,
    M >= 0.1,
    M <= 0.3,
    S >= 0, B >= 0, I >= 0, M >= 0,
]

# 4. Formulate and solve the problem
problem = cp.Problem(objective, constraints)
problem.solve()

# 5. Print the results
print("Status:", problem.status)
print("Objective:", round(100 * problem.value, 1), "% annual return")
print("S:", round(100 * float(S.value), 1), "%")
print("B:", round(100 * float(B.value), 1), "%")
print("I:", round(100 * float(I.value), 1), "%")
print("M:", round(100 * float(M.value), 1), "%")


## 2. Advertising Allocation

A company is planning an advertising campaign using TV, radio, and newspapers. It wants to maximize total exposure. Additional placements in the same medium have smaller exposure and customer acquisition effects than the first block of placements.

| Medium and placement block | Variable | Exposure per placement | New customers per placement | Cost per placement (KRW 10,000) |
|---|---|---:|---:|---:|
| TV: first 10 placements | T1 | 90 | 4,000 | 1,000 |
| TV: additional placements | T2 | 55 | 1,500 | 1,000 |
| Radio: first 15 placements | R1 | 25 | 2,000 | 300 |
| Radio: additional placements | R2 | 20 | 1,200 | 300 |
| Newspaper: first 20 placements | N1 | 10 | 1,000 | 100 |
| Newspaper: additional placements | N2 | 5 | 800 | 100 |

**Campaign requirements**

- Spend no more than KRW 300 million, equivalent to 30,000 in units of KRW 10,000.
- Attract at least 100,000 new customers, using the additive customer estimates in the table.
- Use between 14 and 20 TV placements.
- Use no more than 33 radio placements. The total number of radio placements must be at least twice the total number of TV placements.
- Use at least 30 newspaper placements.
- Limit the first blocks to 10 TV, 15 radio, and 20 newspaper placements. All variables must be nonnegative.

**Your task**

1. Define the six placement variables and formulate an LP that maximizes total exposure.
2. Solve the model using CVXPY and report placements in each block and the total for each medium.
3. Calculate total spending and expected new customers from the unrounded solution, and check the campaign requirements.

Use continuous variables for this LP exercise. Rounded placement counts are for display and may not satisfy all constraints. The objective is measured in exposure units.


In [ ]:
import cvxpy as cp

# 1. Decision variables
T1 = cp.Variable()
T2 = cp.Variable()
R1 = cp.Variable()
R2 = cp.Variable()
N1 = cp.Variable()
N2 = cp.Variable()

# 2. Objective function
objective = cp.Maximize(
    90*T1 + 55*T2 + 25*R1 + 20*R2 + 10*N1 + 5*N2
)

# 3. Constraints
constraints = [
    1000*T1 + 1000*T2 + 300*R1 + 300*R2 + 100*N1 + 100*N2 <= 30000,
    4000*T1 + 1500*T2 + 2000*R1 + 1200*R2 + 1000*N1 + 800*N2 >= 100000,
    R1 + R2 >= 2*T1 + 2*T2,
    T1 + T2 >= 14,
    T1 + T2 <= 20,
    R1 + R2 <= 33,
    N1 + N2 >= 30,
    T1 <= 10,
    R1 <= 15,
    N1 <= 20,
    T1 >= 0, T2 >= 0, R1 >= 0, R2 >= 0,
    N1 >= 0, N2 >= 0,
]

# 4. Formulate and solve the problem
problem = cp.Problem(objective, constraints)
problem.solve()

# 5. Print the results
print("Status:", problem.status)
print("Exposure:", round(problem.value))
print("T1:", round(float(T1.value)))
print("T2:", round(float(T2.value)))
print("R1:", round(float(R1.value)))
print("R2:", round(float(R2.value)))
print("N1:", round(float(N1.value)))
print("N2:", round(float(N2.value)))


In [ ]:
print("Status:", problem.status)
print("Exposure:", round(problem.value))
print("T1:", round(float(T1.value)))
print("T2:", round(float(T2.value)))
print("R1:", round(float(R1.value)))
print("R2:", round(float(R2.value)))
print("N1:", round(float(N1.value)))
print("N2:", round(float(N2.value)))

## 3. Make-or-Buy Decisions

A manufacturer must meet demand for three products. It can produce each product internally or purchase finished units from an outside supplier. Internal production uses limited manufacturing, assembly, and quality-control capacity. Purchased finished units do not use these internal capacities in this model.

| Product | Demand (units) | Internal production cost (KRW/unit) | Purchase cost (KRW/unit) |
|---|---:|---:|---:|
| A | 60,000 | 4,500 | 6,000 |
| B | 40,000 | 5,000 | 7,000 |
| C | 35,000 | 2,500 | 5,000 |

| Internal process | Product A (min/unit) | Product B (min/unit) | Product C (min/unit) | Available capacity (minutes) |
|---|---:|---:|---:|---:|
| Manufacturing | 2 | 3 | 4 | 216,000 |
| Assembly | 1.5 | 2 | 3 | 150,000 |
| Quality control | 2 | 1.5 | 4 | 180,000 |

**Decision rules**

- For each product, internal production plus purchases must be at least its demand.
- Internal production must stay within all three process capacities.
- There is no separate limit on outside purchases.
- All quantities must be nonnegative. Treat them as continuous variables for this LP exercise.

**Your task**

1. Let A, B, and C denote quantities made, and BA, BB, and BC denote quantities bought. Formulate an LP that minimizes total production and purchasing cost.
2. Solve the model using CVXPY and report how much of each product to make and buy.
3. Calculate capacity usage for each process using the unrounded solution. Identify any binding capacity constraints.

The objective uses KRW. The code displays total cost in KRW million and rounds quantities to integers. Use unrounded values when checking costs and capacity usage.


In [ ]:
import cvxpy as cp

# 1. Decision variables
A = cp.Variable()
B = cp.Variable()
C = cp.Variable()
BA = cp.Variable()
BB = cp.Variable()
BC = cp.Variable()

# 2. Objective function
objective = cp.Minimize(
    4500*A + 5000*B + 2500*C + 6000*BA + 7000*BB + 5000*BC
)

# 3. Constraints
constraints = [
    2*A + 3*B + 4*C <= 216000,
    1.5*A + 2*B + 3*C <= 150000,
    2*A + 1.5*B + 4*C <= 180000,
    A + BA >= 60000,
    B + BB >= 40000,
    C + BC >= 35000,
    A >= 0, B >= 0, C >= 0, BA >= 0,
    BB >= 0, BC >= 0,
]

# 4. Formulate and solve the problem
problem = cp.Problem(objective, constraints)
problem.solve()

# 5. Print the results
print("Status:", problem.status)
print("KRW million:", round(problem.value / 1e6))
print("A:", round(float(A.value)))
print("B:", round(float(B.value)))
print("C:", round(float(C.value)))
print("BA:", round(float(BA.value)))
print("BB:", round(float(BB.value)))
print("BC:", round(float(BC.value)))

## 4. Hourly Staff Scheduling

A service operation needs staff from 9 a.m. to 7 p.m. It can hire two types of employees and choose their starting times. Determine a staffing plan that meets hourly demand at minimum daily cost.

| Hour | Minimum staff required |
|---|---:|
| 9–10 a.m. | 5 |
| 10–11 a.m. | 4 |
| 11 a.m.–noon | 9 |
| Noon–1 p.m. | 10 |
| 1–2 p.m. | 9 |
| 2–3 p.m. | 6 |
| 3–4 p.m. | 4 |
| 4–5 p.m. | 7 |
| 5–6 p.m. | 4 |
| 6–7 p.m. | 4 |

| Employee type | Shift pattern | Cost per employee |
|---|---|---:|
| X-type | Work 4 consecutive hours, take a 1-hour break, then work 3 more hours | KRW 128,000 |
| Y-type | Work 4 consecutive hours with no break | KRW 40,000 |

**Scheduling rules**

- Employees may start on the hour at 9, 10, or 11 a.m., noon, or 1, 2, 3, 4, 5, or 6 p.m.
- Meet the total staffing requirement in every hour.
- At least one X-type employee must be working in every hour. An X-type employee on break does not count toward coverage.
- Charge the full shift cost for each employee. The model checks coverage only during the listed hours, including when a late-starting shift extends beyond 7 p.m.
- All staffing variables must be nonnegative. Use continuous variables for this LP exercise.

Let Xh and Yh denote the numbers of X-type and Y-type employees starting at hour h. In the code, labels 1–6 mean afternoon hours and 12 means noon. For example, X9 works from 9 a.m. to 1 p.m., takes a break from 1 to 2 p.m., and returns from 2 to 5 p.m. Y9 works from 9 a.m. to 1 p.m.

**Your task**

1. Define the 20 starting-time variables and formulate the cost-minimizing LP. Write separate constraints for total hourly coverage and X-type hourly coverage.
2. Solve the model using CVXPY and report staffing by starting time and employee type, together with the minimum daily cost.
3. Check hourly coverage using the unrounded solution. Explain why rounding continuous staff counts alone does not guarantee a feasible integer schedule.

The model measures costs in KRW 10,000, so the two cost coefficients are 12.8 and 4. The printed total cost is converted to KRW.


In [ ]:
import cvxpy as cp

# 1. Decision variables
X9 = cp.Variable()
X10 = cp.Variable()
X11 = cp.Variable()
X12 = cp.Variable()
X1 = cp.Variable()
X2 = cp.Variable()
X3 = cp.Variable()
X4 = cp.Variable()
X5 = cp.Variable()
X6 = cp.Variable()
Y9 = cp.Variable()
Y10 = cp.Variable()
Y11 = cp.Variable()
Y12 = cp.Variable()
Y1 = cp.Variable()
Y2 = cp.Variable()
Y3 = cp.Variable()
Y4 = cp.Variable()
Y5 = cp.Variable()
Y6 = cp.Variable()

# 2. Objective function
objective = cp.Minimize(
    12.8*(X9 + X10 + X11 + X12 + X1 + X2 + X3 + X4 + X5 + X6)
    + 4*(Y9 + Y10 + Y11 + Y12 + Y1 + Y2 + Y3 + Y4 + Y5 + Y6)
)

# 3. Constraints
constraints = [
    X9 + Y9 >= 5,
    X9 + X10 + Y9 + Y10 >= 4,
    X9 + X10 + X11 + Y9 + Y10 + Y11 >= 9,
    X9 + X10 + X11 + X12 + Y9 + Y10 + Y11 + Y12 >= 10,
    X10 + X11 + X12 + X1 + Y10 + Y11 + Y12 + Y1 >= 9,
    X9 + X11 + X12 + X1 + X2 + Y11 + Y12 + Y1 + Y2 >= 6,
    X9 + X10 + X12 + X1 + X2 + X3 + Y12 + Y1 + Y2 + Y3 >= 4,
    X9 + X10 + X11 + X1 + X2 + X3 + X4 + Y1 + Y2 + Y3 + Y4 >= 7,
    X10 + X11 + X12 + X2 + X3 + X4 + X5 + Y2 + Y3 + Y4 + Y5 >= 4,
    X11 + X12 + X1 + X3 + X4 + X5 + X6 + Y3 + Y4 + Y5 + Y6 >= 4,
]

constraints += [
    X9 >= 1,
    X9 + X10 >= 1,
    X9 + X10 + X11 >= 1,
    X9 + X10 + X11 + X12 >= 1,
    X10 + X11 + X12 + X1 >= 1,
    X9 + X11 + X12 + X1 + X2 >= 1,
    X9 + X10 + X12 + X1 + X2 + X3 >= 1,
    X9 + X10 + X11 + X1 + X2 + X3 + X4 >= 1,
    X10 + X11 + X12 + X2 + X3 + X4 + X5 >= 1,
    X11 + X12 + X1 + X3 + X4 + X5 + X6 >= 1,
    X9 >= 0, X10 >= 0, X11 >= 0, X12 >= 0,
    X1 >= 0, X2 >= 0, X3 >= 0, X4 >= 0,
    X5 >= 0, X6 >= 0, Y9 >= 0, Y10 >= 0,
    Y11 >= 0, Y12 >= 0, Y1 >= 0, Y2 >= 0,
    Y3 >= 0, Y4 >= 0, Y5 >= 0, Y6 >= 0,
]

# 4. Formulate and solve the problem
problem = cp.Problem(objective, constraints)
problem.solve()

# 5. Print the results
print("Status:", problem.status)
print("Total Cost (unit: KRW 10,000):", round(problem.value))
print("X9:", round(float(X9.value)))
print("X10:", round(float(X10.value)))
print("X11:", round(float(X11.value)))
print("X12:", round(float(X12.value)))
print("X1:", round(float(X1.value)))
print("X2:", round(float(X2.value)))
print("X3:", round(float(X3.value)))
print("X4:", round(float(X4.value)))
print("X5:", round(float(X5.value)))
print("X6:", round(float(X6.value)))
print("Y9:", round(float(Y9.value)))
print("Y10:", round(float(Y10.value)))
print("Y11:", round(float(Y11.value)))
print("Y12:", round(float(Y12.value)))
print("Y1:", round(float(Y1.value)))
print("Y2:", round(float(Y2.value)))
print("Y3:", round(float(Y3.value)))
print("Y4:", round(float(Y4.value)))
print("Y5:", round(float(Y5.value)))
print("Y6:", round(float(Y6.value)))

## 5. Airline Ticket Allocation

An airline operates four flight legs through Cheongju: Beijing–Cheongju, Shanghai–Cheongju, Cheongju–Jeju, and Cheongju–Tokyo. Each flight leg has 132 seats. The airline sells full-fare and discount tickets for eight origin–destination routes and wants to maximize total ticket revenue.

Passengers traveling from Beijing or Shanghai to Jeju or Tokyo connect at Cheongju. Each connecting passenger uses one seat on each of the two flight legs. The fare below is for the entire origin–destination trip.

| Route | Full fare (KRW 10,000) | Discount fare (KRW 10,000) | Full-fare demand | Discount demand | Variables (full / discount) |
|---|---:|---:|---:|---:|---|
| Beijing–Cheongju | 20 | 13 | 16 | 33 | BCF / BCD |
| Beijing–Jeju | 30 | 19 | 6 | 44 | BJF / BJD |
| Beijing–Tokyo | 31 | 21 | 11 | 45 | BTF / BTD |
| Shanghai–Cheongju | 21 | 14 | 15 | 26 | SCF / SCD |
| Shanghai–Tokyo | 31 | 23 | 9 | 39 | STF / STD |
| Shanghai–Jeju | 29 | 19 | 7 | 56 | SJF / SJD |
| Cheongju–Jeju | 9 | 7 | 8 | 64 | CJF / CJD |
| Cheongju–Tokyo | 25 | 18 | 10 | 46 | CTF / CTD |

**Sales and capacity rules**

- Ticket sales for each route and fare class cannot exceed the corresponding demand. Demand is an upper bound, not a sales requirement.
- Total passengers using any flight leg cannot exceed 132.
- Beijing–Cheongju carries all passengers whose journeys start in Beijing, including connecting passengers.
- Shanghai–Cheongju carries all passengers whose journeys start in Shanghai, including connecting passengers.
- Cheongju–Jeju carries all passengers whose journeys end in Jeju, including passengers connecting from Beijing or Shanghai.
- Cheongju–Tokyo carries all passengers whose journeys end in Tokyo, including passengers connecting from Beijing or Shanghai.
- All ticket quantities must be nonnegative. Use continuous variables for this LP exercise.

**Your task**

1. Define the 16 ticket-sales variables and formulate an LP that maximizes revenue, with demand bounds and four shared-capacity constraints.
2. Solve the model using CVXPY and report ticket sales by route and fare class, along with total revenue.
3. Calculate the number of seats used on each flight leg and identify any unmet ticket demand.

F denotes full fare and D denotes discount fare. The objective and printed revenue use units of KRW 10,000. Use unrounded values when checking revenue and seat usage.


In [ ]:
import cvxpy as cp

# 1. Decision variables
BCF = cp.Variable()
BCD = cp.Variable()
BJF = cp.Variable()
BJD = cp.Variable()
BTF = cp.Variable()
BTD = cp.Variable()
SCF = cp.Variable()
SCD = cp.Variable()
STF = cp.Variable()
STD = cp.Variable()
SJF = cp.Variable()
SJD = cp.Variable()
CJF = cp.Variable()
CJD = cp.Variable()
CTF = cp.Variable()
CTD = cp.Variable()

# 2. Objective function
objective = cp.Maximize(
    20*BCF + 13*BCD + 30*BJF + 19*BJD + 31*BTF + 21*BTD
    + 21*SCF + 14*SCD + 31*STF + 23*STD + 29*SJF + 19*SJD
    + 9*CJF + 7*CJD + 25*CTF + 18*CTD
)

# 3. Constraints
constraints = [
    BCF <= 16,
    BCD <= 33,
    BJF <= 6,
    BJD <= 44,
    BTF <= 11,
    BTD <= 45,
    SCF <= 15,
    SCD <= 26,
    STF <= 9,
    STD <= 39,
    SJF <= 7,
    SJD <= 56,
    CJF <= 8,
    CJD <= 64,
    CTF <= 10,
    CTD <= 46,
]

constraints += [
    BCF + BCD + BJF + BJD + BTF + BTD <= 132,
    SCF + SCD + STF + STD + SJF + SJD <= 132,
    BJF + BJD + SJF + SJD + CJF + CJD <= 132,
    BTF + BTD + STF + STD + CTF + CTD <= 132,
    BCF >= 0, BCD >= 0, BJF >= 0, BJD >= 0,
    BTF >= 0, BTD >= 0, SCF >= 0, SCD >= 0,
    STF >= 0, STD >= 0, SJF >= 0, SJD >= 0,
    CJF >= 0, CJD >= 0, CTF >= 0, CTD >= 0,
]

# 4. Formulate and solve the problem
problem = cp.Problem(objective, constraints)
problem.solve()

# 5. Print the results
print("Status:", problem.status)
print("Revenue (unit: KRW 10,000):", round(problem.value))
print("BCF:", round(float(BCF.value)))
print("BCD:", round(float(BCD.value)))
print("BJF:", round(float(BJF.value)))
print("BJD:", round(float(BJD.value)))
print("BTF:", round(float(BTF.value)))
print("BTD:", round(float(BTD.value)))
print("SCF:", round(float(SCF.value)))
print("SCD:", round(float(SCD.value)))
print("STF:", round(float(STF.value)))
print("STD:", round(float(STD.value)))
print("SJF:", round(float(SJF.value)))
print("SJD:", round(float(SJD.value)))
print("CJF:", round(float(CJF.value)))
print("CJD:", round(float(CJD.value)))
print("CTF:", round(float(CTF.value)))
print("CTD:", round(float(CTD.value)))